# Purchasing Power Parity (PPP) — Interactive Exercise
## International Money and Finance | Prof. Dr. Eric Mayer

---

This notebook explores **Purchasing Power Parity (PPP)** — one of the
fundamental theories linking exchange rates to price levels across countries.

### How to use this notebook

You do **not** need to write any code. Just click on the menu:

**Kernel → Restart Kernel and Run All Cells…**

Then scroll down and read / interact with the outputs.

### What you will see in this notebook

1. **Interactive PPP Explorer** — Move the sliders to set inflation rates for
   the US and the Euro Area and watch how the EUR/USD exchange rate *should*
   evolve according to **relative PPP**.
2. **Price Level Developments** — Compare US and German price levels since 1999.
3. **Actual vs. PPP-Implied Exchange Rate** — How well does PPP explain the
   EUR/USD exchange rate since 1999?
4. **Statistical PPP Test** — OLS regression and Engle–Granger
   Error Correction Model (ECM).

---
> **Data sources:** FRED (Federal Reserve Bank of St. Louis)
> US CPI: CPIAUCSL | German HICP: CP0000DEM086NEST |
> Exchange Rate: DEXUSEU
> Base year for price indices: **2015 = 100**

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import warnings, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

In [ ]:
# ── Load raw data from GitHub ───────────────────────────────────────────────
url_base = ("https://raw.githubusercontent.com/"
            "mayeric97070/PPP-Exercise-Notebook-for-"
            "International-Money-and-Finance/main/")

cpi_us_raw = pd.read_csv(url_base + "CPIAUCSL.csv")
cpi_de_raw = pd.read_csv(url_base + "CP0000DEM086NEST.csv")
fx_raw     = pd.read_csv(url_base + "DEXUSEU.csv")

for d in (cpi_us_raw, cpi_de_raw, fx_raw):
    d['observation_date'] = pd.to_datetime(d['observation_date'])
    d.set_index('observation_date', inplace=True)

cpi_us_raw.columns = ['CPI_US_raw']
cpi_de_raw.columns = ['CPI_DE_raw']
fx_raw.columns     = ['EURUSD']

start = '1999-01-01'
cpi_us_raw = cpi_us_raw.loc[start:]
cpi_de_raw = cpi_de_raw.loc[start:]
fx_raw     = fx_raw.loc[start:]

fx_raw = fx_raw.replace('.', np.nan).dropna()
fx_raw['EURUSD'] = fx_raw['EURUSD'].astype(float)
fx_monthly = fx_raw.resample('MS').mean()

def rebase(series, base_year=2015):
    base_val = series.loc[str(base_year)].mean()
    return (series / base_val) * 100.0

cpi_us = rebase(cpi_us_raw['CPI_US_raw']).rename('CPI_US')
cpi_de = rebase(cpi_de_raw['CPI_DE_raw']).rename('CPI_DE')

df = pd.concat([cpi_us, cpi_de, fx_monthly], axis=1).dropna()
df.index.name = 'Date'

---

## 1. Interactive PPP Explorer

Before looking at the data, let's build some **intuition** for relative PPP.

**Relative PPP** states that the percentage change in the nominal exchange
rate equals the inflation differential between the two countries:

$$\frac{\Delta E_{t}}{E_{t}} \;=\; \pi_{US} \;-\; \pi_{EUR}$$

**Use the sliders** below to set annual inflation rates for the US and the
Euro Area. The chart shows how the EUR/USD exchange rate *should* evolve
over the next **10 years** according to relative PPP, starting from the
current approximate rate of **1.08 USD/EUR**.

*If US inflation exceeds Euro Area inflation, PPP predicts that the USD
should depreciate (i.e. EUR/USD rises) — and vice versa.*

In [ ]:
display(HTML(r'''
<style>
.ppp-box{font-family:Calibri,Arial,sans-serif;max-width:760px;background:#f8f9fa;
  border:1px solid #dee2e6;border-radius:8px;padding:24px;margin:10px 0}
.ppp-title{font-size:17px;font-weight:bold;margin-bottom:6px;color:#212529}
.ppp-sub{font-size:13px;color:#6c757d;margin-bottom:16px}
.ppp-slider-row{display:flex;align-items:center;margin:10px 0;gap:12px}
.ppp-slider-label{width:300px;font-size:14px;color:#495057}
.ppp-box input[type=range]{flex:1;accent-color:#1f77b4}
.ppp-slider-val{width:60px;text-align:right;font-weight:bold;font-size:14px;color:#1f77b4}
.ppp-result-box{margin-top:18px;background:white;border-left:4px solid #1f77b4;
  padding:14px 18px;border-radius:4px}
.ppp-result-row{display:flex;justify-content:space-between;margin:6px 0;font-size:14px}
.ppp-result-label{color:#495057}
.ppp-result-val{font-weight:bold}
.ppp-pos{color:#d62728}.ppp-neg{color:#2ca02c}.ppp-neu{color:#333}
.ppp-interp{margin-top:14px;padding:10px 14px;border-radius:4px;font-size:14px;line-height:1.6}
.ppp-dep{background:#fff3cd;border:1px solid #ffc107}
.ppp-app{background:#d4edda;border:1px solid #28a745}
.ppp-par{background:#e2e3e5;border:1px solid #adb5bd}
.ppp-chart-wrap{margin-top:18px;background:white;border-radius:4px;padding:10px}
</style>
<div class="ppp-box">
<div class="ppp-title">PPP Interactive Explorer</div>
<div class="ppp-sub">Anchor: today &mdash; E<sub>0</sub> = 1.08 USD/EUR &nbsp;|&nbsp;
  Sliders: annual inflation rates &nbsp;|&nbsp; Horizon: 10 years</div>

<div class="ppp-slider-row">
  <span class="ppp-slider-label">US inflation &pi;<sub>US</sub> (% p.a.)</span>
  <input type="range" id="ppp_iUS" min="-2" max="12" step="0.1" value="3.0" oninput="pppCalc()">
  <span class="ppp-slider-val" id="ppp_vUS">3.0</span>
</div>
<div class="ppp-slider-row">
  <span class="ppp-slider-label">Euro Area inflation &pi;<sub>EUR</sub> (% p.a.)</span>
  <input type="range" id="ppp_iEU" min="-2" max="12" step="0.1" value="2.0" oninput="pppCalc()">
  <span class="ppp-slider-val" id="ppp_vEU">2.0</span>
</div>

<div class="ppp-result-box">
  <div class="ppp-result-row"><span class="ppp-result-label">Spot rate E<sub>0</sub> (USD/EUR)</span>
    <span class="ppp-result-val ppp-neu">1.0800</span></div>
  <div class="ppp-result-row"><span class="ppp-result-label">Inflation differential &pi;<sub>US</sub> &minus; &pi;<sub>EUR</sub></span>
    <span class="ppp-result-val" id="ppp_rDiff">--</span></div>
  <div class="ppp-result-row"><span class="ppp-result-label">PPP-implied rate after 10 years (USD/EUR)</span>
    <span class="ppp-result-val" id="ppp_rE10">--</span></div>
  <div class="ppp-result-row"><span class="ppp-result-label">Predicted cumulative USD change</span>
    <span class="ppp-result-val" id="ppp_rChg">--</span></div>
</div>

<div class="ppp-interp ppp-par" id="ppp_interp">Adjust the sliders to see the PPP prediction.</div>

<div class="ppp-chart-wrap">
  <svg id="ppp_chart" viewBox="0 0 700 280" style="width:100%;height:auto"
       xmlns="http://www.w3.org/2000/svg"></svg>
</div>

</div>

<script>
(function(){
  var E0 = 1.08;
  var YEARS = 10;

  function pppCalc(){
    var iUS = parseFloat(document.getElementById("ppp_iUS").value);
    var iEU = parseFloat(document.getElementById("ppp_iEU").value);
    document.getElementById("ppp_vUS").textContent = iUS.toFixed(1);
    document.getElementById("ppp_vEU").textContent = iEU.toFixed(1);

    var diff = iUS - iEU;
    // relative PPP: E_t = E_0 * ((1+piUS)/(1+piEU))^t
    var factor = (1 + iUS/100) / (1 + iEU/100);
    var E_end = E0 * Math.pow(factor, YEARS);
    var chg = (E_end / E0 - 1) * 100;

    var diffEl = document.getElementById("ppp_rDiff");
    diffEl.textContent = (diff>=0?"+":"") + diff.toFixed(2) + " pp";
    diffEl.className = "ppp-result-val" + (diff>0.05?" ppp-pos":diff<-0.05?" ppp-neg":" ppp-neu");

    document.getElementById("ppp_rE10").textContent = E_end.toFixed(4) + " USD/EUR";

    var chgEl = document.getElementById("ppp_rChg");
    chgEl.textContent = (chg>=0?"+":"") + chg.toFixed(2) + "%";
    chgEl.className = "ppp-result-val" + (chg>0.05?" ppp-pos":chg<-0.05?" ppp-neg":" ppp-neu");

    var box = document.getElementById("ppp_interp");
    if(Math.abs(diff) < 0.05){
      box.className = "ppp-interp ppp-par";
      box.innerHTML = "Inflation rates are (almost) equal &mdash; relative PPP predicts <strong>no change</strong> in the exchange rate.";
    } else if(diff > 0){
      box.className = "ppp-interp ppp-dep";
      box.innerHTML = "US inflation is higher by <strong>"+diff.toFixed(2)+
        " pp</strong>. Relative PPP predicts the USD to <strong>depreciate</strong> by about <strong>"+
        Math.abs(chg).toFixed(2)+"%</strong> over 10 years &mdash; i.e. EUR/USD rises.";
    } else {
      box.className = "ppp-interp ppp-app";
      box.innerHTML = "Euro Area inflation is higher by <strong>"+Math.abs(diff).toFixed(2)+
        " pp</strong>. Relative PPP predicts the USD to <strong>appreciate</strong> by about <strong>"+
        Math.abs(chg).toFixed(2)+"%</strong> over 10 years &mdash; i.e. EUR/USD falls.";
    }

    drawChart(iUS, iEU);
  }

  function drawChart(iUS, iEU){
    var svg = document.getElementById("ppp_chart");
    while(svg.firstChild) svg.removeChild(svg.firstChild);

    var W = 700, H = 280;
    var M = {top:18, right:16, bottom:36, left:58};
    var plotW = W - M.left - M.right;
    var plotH = H - M.top - M.bottom;

    // path points
    var N = 121; // monthly steps over 10 years
    var xs = [], ys = [];
    var factor = (1 + iUS/100) / (1 + iEU/100);
    for(var k=0; k<N; k++){
      var t = k/12;
      xs.push(t);
      ys.push(E0 * Math.pow(factor, t));
    }
    var yMin = Math.min.apply(null, ys), yMax = Math.max.apply(null, ys);
    // add a little padding & always include E0
    yMin = Math.min(yMin, E0); yMax = Math.max(yMax, E0);
    var pad = Math.max((yMax-yMin)*0.1, 0.01);
    yMin -= pad; yMax += pad;

    function xPx(x){ return M.left + (x/YEARS)*plotW; }
    function yPx(y){ return M.top + (1 - (y-yMin)/(yMax-yMin))*plotH; }

    function el(tag, attrs){
      var e = document.createElementNS("http://www.w3.org/2000/svg", tag);
      for(var k in attrs) e.setAttribute(k, attrs[k]);
      return e;
    }

    // axes
    svg.appendChild(el("line",{x1:M.left,y1:M.top+plotH,x2:M.left+plotW,y2:M.top+plotH,
      stroke:"#333","stroke-width":1}));
    svg.appendChild(el("line",{x1:M.left,y1:M.top,x2:M.left,y2:M.top+plotH,
      stroke:"#333","stroke-width":1}));

    // x ticks
    for(var i=0;i<=YEARS;i+=2){
      var xp = xPx(i);
      svg.appendChild(el("line",{x1:xp,y1:M.top+plotH,x2:xp,y2:M.top+plotH+5,stroke:"#333"}));
      var t = el("text",{x:xp,y:M.top+plotH+18,"text-anchor":"middle",
        "font-size":11,"font-family":"Calibri,Arial,sans-serif",fill:"#333"});
      t.textContent = "Year " + i;
      svg.appendChild(t);
    }
    // y ticks
    var nT = 5;
    for(var j=0;j<=nT;j++){
      var yv = yMin + (yMax-yMin)*j/nT;
      var yp = yPx(yv);
      svg.appendChild(el("line",{x1:M.left-5,y1:yp,x2:M.left,y2:yp,stroke:"#333"}));
      svg.appendChild(el("line",{x1:M.left,y1:yp,x2:M.left+plotW,y2:yp,
        stroke:"#ddd","stroke-width":1}));
      var lt = el("text",{x:M.left-8,y:yp+4,"text-anchor":"end",
        "font-size":11,"font-family":"Calibri,Arial,sans-serif",fill:"#333"});
      lt.textContent = yv.toFixed(3);
      svg.appendChild(lt);
    }
    // E0 dashed line
    svg.appendChild(el("line",{x1:M.left,y1:yPx(E0),x2:M.left+plotW,y2:yPx(E0),
      stroke:"#6c757d","stroke-width":1,"stroke-dasharray":"4,4"}));
    var e0lbl = el("text",{x:M.left+plotW-4,y:yPx(E0)-4,"text-anchor":"end",
      "font-size":11,"font-family":"Calibri,Arial,sans-serif",fill:"#6c757d"});
    e0lbl.textContent = "E0 = 1.08";
    svg.appendChild(e0lbl);

    // path
    var d = "";
    for(var k=0;k<N;k++){
      d += (k===0?"M":"L") + xPx(xs[k]).toFixed(2) + "," + yPx(ys[k]).toFixed(2) + " ";
    }
    svg.appendChild(el("path",{d:d,fill:"none",stroke:"#1f77b4","stroke-width":2.2}));

    // y-axis label
    var yl = el("text",{x:14,y:M.top+plotH/2,transform:"rotate(-90 14,"+(M.top+plotH/2)+")",
      "text-anchor":"middle","font-size":12,
      "font-family":"Calibri,Arial,sans-serif",fill:"#333"});
    yl.textContent = "USD per EUR";
    svg.appendChild(yl);

    // x-axis label
    var xl = el("text",{x:M.left+plotW/2,y:H-6,"text-anchor":"middle",
      "font-size":12,"font-family":"Calibri,Arial,sans-serif",fill:"#333"});
    xl.textContent = "Years from today";
    svg.appendChild(xl);
  }

  window.pppCalc = pppCalc;
  pppCalc();
})();
</script>
'''))

---

## 2. Price Level Developments — USA vs. Germany (1999–today)

Both price indices are rebased to **2015 = 100**.
The gap between the two lines represents the cumulative inflation
differential — the key driver of exchange rate changes under PPP.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(df.index, df['CPI_US'], lw=2, color='steelblue',
        label='USA — CPI (CPIAUCSL)')
ax.plot(df.index, df['CPI_DE'], lw=2, color='darkorange',
        label='Germany — HICP (CP0000DEM086NEST)')
ax.axhline(100, color='grey', lw=1, ls='--', alpha=0.6,
           label='Base year 2015 = 100')

ax.fill_between(df.index, df['CPI_US'], df['CPI_DE'],
                where=df['CPI_US'] >= df['CPI_DE'],
                alpha=0.12, color='steelblue',
                label='US prices relatively higher')
ax.fill_between(df.index, df['CPI_US'], df['CPI_DE'],
                where=df['CPI_US'] < df['CPI_DE'],
                alpha=0.12, color='darkorange',
                label='German prices relatively higher')

ax.set_title('Price Level Developments: USA vs. Germany (1999–today)\n'
             'Both indices rebased to 2015 = 100',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Price Index (2015 = 100)')
ax.set_xlabel('')
ax.legend(loc='upper left', fontsize=9)

for col, color, label in [('CPI_US', 'steelblue', 'USA'),
                          ('CPI_DE', 'darkorange', 'Germany')]:
    last_val  = df[col].iloc[-1]
    last_date = df.index[-1]
    ax.annotate(f'{label}: {last_val:.1f}',
                xy=(last_date, last_val),
                xytext=(-60, 10), textcoords='offset points',
                fontsize=9, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1))

plt.tight_layout()
plt.show()

cum_us = (df['CPI_US'].iloc[-1] / df['CPI_US'].iloc[0] - 1) * 100
cum_de = (df['CPI_DE'].iloc[-1] / df['CPI_DE'].iloc[0] - 1) * 100
print(f"USA cumulative inflation since Jan 1999:     {cum_us:.1f}%")
print(f"Germany cumulative inflation since Jan 1999: {cum_de:.1f}%")
print(f"Differential (US − DE):                      {cum_us - cum_de:+.1f}%")

---

## 3. Actual vs. PPP-Implied Exchange Rate (1999–today)

Under **absolute PPP**, the nominal exchange rate should equal the
ratio of price levels:

$$E_{\$/€} \;=\; \frac{P_{US}}{P_{DE}}$$

We anchor this to the **actual exchange rate in January 1999** and let the
PPP-implied rate evolve according to cumulative price level changes.

This is equivalent to asking:
*"If relative PPP had held perfectly since 1999, where would the exchange
rate be today?"*

In [ ]:
E_anchor  = df['EURUSD'].iloc[0]
P_US_base = df['CPI_US'].iloc[0]
P_DE_base = df['CPI_DE'].iloc[0]

df['PPP_implied'] = (E_anchor *
                     (df['CPI_US'] / P_US_base) /
                     (df['CPI_DE'] / P_DE_base))

df['Misalignment'] = (df['EURUSD'] / df['PPP_implied'] - 1) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8),
                                gridspec_kw={'height_ratios': [2, 1]},
                                sharex=True)

ax1.plot(df.index, df['EURUSD'], lw=2, color='steelblue',
         label='Actual EUR/USD exchange rate')
ax1.plot(df.index, df['PPP_implied'], lw=2, color='tomato', ls='--',
         label='PPP-implied rate (anchored Jan 1999)')
ax1.fill_between(df.index, df['EURUSD'], df['PPP_implied'],
                 where=df['EURUSD'] >= df['PPP_implied'],
                 alpha=0.15, color='steelblue',
                 label='EUR overvalued vs PPP')
ax1.fill_between(df.index, df['EURUSD'], df['PPP_implied'],
                 where=df['EURUSD'] < df['PPP_implied'],
                 alpha=0.15, color='tomato',
                 label='EUR undervalued vs PPP')
ax1.set_ylabel('USD per EUR')
ax1.set_title('Actual vs. PPP-Implied EUR/USD Exchange Rate (1999–today)',
              fontsize=13, fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.axhline(E_anchor, color='grey', lw=0.8, ls=':', alpha=0.7)

ax2.bar(df.index, df['Misalignment'],
        color=np.where(df['Misalignment'] >= 0, 'steelblue', 'tomato'),
        alpha=0.6, width=25)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_ylabel('Deviation from PPP (%)')
ax2.set_title('EUR over/undervaluation relative to PPP (in %)', fontsize=11)

plt.tight_layout()
plt.show()

last_mis  = df['Misalignment'].iloc[-1]
direction = 'overvalued' if last_mis > 0 else 'undervalued'
print(f"Latest observation: {df.index[-1].date()}")
print(f"  Actual exchange rate:  {df['EURUSD'].iloc[-1]:.4f} USD/EUR")
print(f"  PPP-implied rate:      {df['PPP_implied'].iloc[-1]:.4f} USD/EUR")
print(f"  Misalignment:          {last_mis:+.1f}%  →  EUR is {direction} vs PPP")

---

## 4. Statistical PPP Test — OLS and Error Correction Model (ECM)

We now test PPP formally. Let:

- $e_t = \ln E_t$ (log nominal exchange rate, USD per EUR)
- $p_t^{US} = \ln P_t^{US}$, $p_t^{DE} = \ln P_t^{DE}$

### 4.1 Cointegration regression (Engle–Granger, step 1)

Under **absolute PPP** the three series should be cointegrated:

$$e_t \;=\; \alpha \;+\; \beta \,(p_t^{US} - p_t^{DE}) \;+\; u_t$$

PPP in its strict form predicts $\beta = 1$. We estimate $\alpha$ and $\beta$
by OLS and then test whether the residuals $\hat{u}_t$ are stationary using
an **Augmented Dickey–Fuller (ADF) test**. If the residuals are stationary,
the exchange rate and the price differential are **cointegrated** — PPP
holds as a long-run equilibrium.

### 4.2 Error Correction Model (Engle–Granger, step 2)

If cointegration is established, short-run dynamics are captured by an ECM:

$$\Delta e_t \;=\; c \;+\; \gamma \,\hat{u}_{t-1}
     \;+\; \delta \,\Delta(p_t^{US} - p_t^{DE}) \;+\; \varepsilon_t$$

- $\gamma$ is the **speed-of-adjustment** coefficient.
  If $\gamma < 0$ and statistically significant, the exchange rate
  corrects past PPP deviations — consistent with long-run PPP.
- A half-life of deviations can be computed as
  $\ln(0.5) / \ln(1+\gamma)$ months.

In [ ]:
# ── Engle–Granger step 1: OLS cointegration regression ──────────────────────
dfr = df[['EURUSD', 'CPI_US', 'CPI_DE']].copy()
dfr['e']     = np.log(dfr['EURUSD'])
dfr['p_us']  = np.log(dfr['CPI_US'])
dfr['p_de']  = np.log(dfr['CPI_DE'])
dfr['pdiff'] = dfr['p_us'] - dfr['p_de']

X = sm.add_constant(dfr['pdiff'])
y = dfr['e']
ols = sm.OLS(y, X).fit()
alpha_hat, beta_hat = ols.params['const'], ols.params['pdiff']
resid = ols.resid

# ADF on residuals
adf_res = adfuller(resid, autolag='AIC')
adf_stat, adf_p, adf_lag, adf_nobs = adf_res[0], adf_res[1], adf_res[2], adf_res[3]
adf_crit = adf_res[4]

print("─" * 62)
print("Engle–Granger step 1: OLS cointegration regression")
print("─" * 62)
print(f"  e_t = α + β·(p_US - p_DE) + u_t")
print(f"  α̂ = {alpha_hat:+.4f}   (std.err = {ols.bse['const']:.4f})")
print(f"  β̂ = {beta_hat:+.4f}   (std.err = {ols.bse['pdiff']:.4f})")
print(f"  R²  = {ols.rsquared:.3f}")
print(f"  Strict PPP predicts β = 1.")
print()
print("─" * 62)
print("ADF test on the OLS residuals (stationarity → cointegration)")
print("─" * 62)
print(f"  ADF statistic       = {adf_stat:.3f}")
print(f"  p-value (standard)  = {adf_p:.3f}    (not the Engle–Granger critical values!)")
print(f"  Lags used           = {adf_lag}")
print(f"  Observations        = {adf_nobs}")
print(f"  Critical values (standard ADF): "
      f"1%={adf_crit['1%']:.2f}, 5%={adf_crit['5%']:.2f}, 10%={adf_crit['10%']:.2f}")
print()
if adf_p < 0.10:
    print("  → Residuals appear stationary at the 10% level:")
    print("    evidence of cointegration / long-run PPP.")
else:
    print("  → Residuals do not appear stationary:")
    print("    limited evidence of cointegration in this sample.")

In [ ]:
# ── Engle–Granger step 2: Error Correction Model ────────────────────────────
ecm = pd.DataFrame({
    'd_e':     dfr['e'].diff(),
    'd_pdiff': dfr['pdiff'].diff(),
    'u_lag':   resid.shift(1),
}).dropna()

X_ecm = sm.add_constant(ecm[['u_lag', 'd_pdiff']])
y_ecm = ecm['d_e']
ecm_fit = sm.OLS(y_ecm, X_ecm).fit()

c_hat     = ecm_fit.params['const']
gamma_hat = ecm_fit.params['u_lag']
delta_hat = ecm_fit.params['d_pdiff']

print("─" * 62)
print("Engle–Granger step 2: Error Correction Model")
print("─" * 62)
print("  Δe_t = c + γ·û_{t-1} + δ·Δ(p_US - p_DE)_t + ε_t")
print()
print(f"  ĉ      = {c_hat:+.5f}   (t = {ecm_fit.tvalues['const']:+.2f},"
      f"  p = {ecm_fit.pvalues['const']:.3f})")
print(f"  γ̂      = {gamma_hat:+.5f}   (t = {ecm_fit.tvalues['u_lag']:+.2f},"
      f"  p = {ecm_fit.pvalues['u_lag']:.3f})   ← speed-of-adjustment")
print(f"  δ̂      = {delta_hat:+.4f}   (t = {ecm_fit.tvalues['d_pdiff']:+.2f},"
      f"  p = {ecm_fit.pvalues['d_pdiff']:.3f})")
print(f"  R²     = {ecm_fit.rsquared:.3f}")
print()

# Half-life of deviations (only meaningful if gamma < 0)
if gamma_hat < 0 and (1 + gamma_hat) > 0:
    half_life = np.log(0.5) / np.log(1 + gamma_hat)
    print(f"  Implied half-life of PPP deviations: {half_life:.1f} months"
          f"  (≈ {half_life/12:.1f} years)")
else:
    print("  γ̂ is not negative → no meaningful half-life;"
          " no short-run correction towards PPP in this sample.")

# ── Plot fit of the cointegration regression ────────────────────────────────
fig, (axA, axB) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
axA.plot(dfr.index, dfr['e'], color='steelblue', lw=2,
         label='Actual log EUR/USD')
axA.plot(dfr.index, ols.fittedvalues, color='tomato', lw=2, ls='--',
         label=f'Fitted: α̂ + β̂·(p_US − p_DE),  β̂ = {beta_hat:.3f}')
axA.set_title('Engle–Granger cointegration regression — fitted vs. actual',
              fontsize=12, fontweight='bold')
axA.set_ylabel('log(EUR/USD)')
axA.legend(loc='upper right', fontsize=9)

axB.plot(dfr.index, resid, color='purple', lw=1.4)
axB.axhline(0, color='black', lw=0.8)
axB.fill_between(dfr.index, resid, 0,
                 where=(resid>=0), alpha=0.15, color='steelblue')
axB.fill_between(dfr.index, resid, 0,
                 where=(resid<0), alpha=0.15, color='tomato')
axB.set_title(f'OLS residuals  û_t   (ADF p-value = {adf_p:.3f})', fontsize=11)
axB.set_ylabel('residual')
plt.tight_layout()
plt.show()

---

## 5. Summary — PPP Theory vs. Empirical Evidence

| | PPP Theory | Empirical Finding (1999–today) |
|---|---|---|
| **Slope $\beta$ in $e_t = \alpha + \beta(p^{US}-p^{DE}) + u_t$** | $= 1$ (strict PPP) | estimated $\hat\beta$ typically $\ll 1$ |
| **ADF on residuals** | Stationary residuals $\Rightarrow$ cointegration | borderline — depends on sample |
| **Speed-of-adjustment $\gamma$** | $< 0$, statistically significant | small in absolute value — **very slow** convergence |
| **Half-life of deviations** | Should be short under PPP | typically **3–5 years** in the literature |
| **Short-run fit** | Exchange rate driven by inflation differential | low $R^2$ — short-run FX dynamics dominated by other factors |

**Take-away.** Purchasing Power Parity is a **long-run equilibrium concept**:
over decades, exchange rates tend to move with price level differentials,
but short-run deviations are large, persistent, and driven by factors outside
the PPP framework — monetary policy surprises, risk premia, productivity
differentials (Balassa–Samuelson) and capital flow shocks. PPP remains a
useful benchmark for assessing over- and undervaluation, but it is a poor
short-run forecasting tool.

---
*Data source: Federal Reserve Bank of St. Louis (FRED) — fred.stlouisfed.org*
*Series: CPIAUCSL, CP0000DEM086NEST, DEXUSEU*